# Qwen2.5-VL-3B (QLoRA) — Sidewalk VQA tiếng Việt — DDP 2× T4

**Thứ tự chạy:**
1. **Cell 1** — cài thư viện → **Restart Session**.
2. **Cell 2** — ghi script `qwen_train.py` ra đĩa.
3. **Cell 3** — `torchrun` train trên **cả 2 GPU** rồi eval.
4. **Cell 4** — đóng gói model + kết quả thành `.zip` để tải về.

> Bật **GPU T4 ×2** ở Settings → Accelerator. Kiểm tra path bằng `!ls /kaggle/input/`.

## Cell 1 — Cài thư viện rồi **Restart Session**

In [1]:
!pip install -q -U "transformers>=4.49.0" "accelerate>=0.34.0" qwen-vl-utils "bitsandbytes>=0.46.1" peft tqdm
print("Cai xong. BAY GIO: Run -> Restart Session, roi chay Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 97.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 57.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.2

## Cell 2 — Ghi script train ra đĩa

In [2]:
%%writefile qwen_train.py
# ============================================================
# Qwen2.5-VL-3B-Instruct (QLoRA) — Sidewalk VQA tiếng Việt — DDP 2x T4
# CHẠY:  !torchrun --nproc_per_node=2 qwen_train.py
#
# Vì sao Qwen2.5-VL-3B:
#   - Đa ngôn ngữ tốt (gồm tiếng Việt) -> hợp data của bạn hơn LLaVA.
#   - Chỉ 3B -> train ~3-4x nhanh hơn LLaVA-7B.
#   - Dynamic resolution: chỉnh MIN/MAX_PIXELS để GIẢM số image token
#     (nút thắt FLOPs lớn nhất) — thứ LLaVA cố định 576 không cho làm.
# ============================================================

import os, re, json, sys
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor,
    BitsAndBytesConfig, TrainingArguments, Trainer,
)
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import warnings
warnings.filterwarnings("ignore")

# ── DDP env (torchrun tự set) ────────────────────────────────
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
RANK       = int(os.environ.get("RANK", 0))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))

# QUAN TRỌNG cho torchrun: ép tạo CUDA context cho GPU của process NÀY trước khi
# load model. Nếu không, from_pretrained (caching_allocator_warmup của transformers
# mới) báo "Cannot access accelerator device when none is available".
if torch.cuda.is_available():
    torch.cuda.set_device(LOCAL_RANK)
    _ = torch.zeros(1, device=f"cuda:{LOCAL_RANK}")   # chạm GPU 1 lần để init context

def rprint(*a, **k):
    if RANK == 0:
        print(*a, **k)

# ── Config ──────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGES_DIR = "/kaggle/input/datasets/dngomnh/sidewalk/images_v2"
TRAIN_CSV  = "/kaggle/input/datasets/dngomnh/sidewalkvqa/train.csv"
TEST_CSV   = "/kaggle/input/datasets/dngomnh/sidewalkvqa/test.csv"
OUT_DIR    = "/kaggle/working/qwen_lora"
OUT_CSV    = "/kaggle/working/results_qwen_ft.csv"
CKPT_CSV   = "/kaggle/working/ckpt_qwen.csv"
DEVICE     = f"cuda:{LOCAL_RANK}"

# Giảm image token để tăng tốc: token ảnh ≈ (H*W)/(28*28), kẹp trong [MIN, MAX]
MIN_PIXELS = 64  * 28 * 28      # >= ~64 image token
MAX_PIXELS = 256 * 28 * 28      # <= ~256 image token (LLaVA cố định 576)

# LoRA + training hyper-params
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
TRAIN_EPOCHS  = 1
BATCH_SIZE    = 4       # per-GPU (3B nhỏ, để batch lớn hơn)
GRAD_ACCUM    = 1       # effective batch = WORLD_SIZE(2) x 4 x 1 = 8
LR            = 2e-4
MAX_TRAIN_ROWS = None   # None = full dataset
MAX_IMG_CACHE = 100

# ── Prompt helpers (giống mạch LLaVA để so sánh công bằng) ────
def build_prompt(question: str, qtype: str) -> str:
    if qtype in ("yes_no", "spatial"):
        inst = "Quan sát ảnh và trả lời câu hỏi sau chỉ bằng 'có' hoặc 'không', không giải thích."
    elif qtype == "how_many":
        inst = "Quan sát ảnh và trả lời câu hỏi sau chỉ bằng một số nguyên, không giải thích."
    else:
        inst = ("Quan sát ảnh và trả lời câu hỏi sau. "
                "Chỉ chọn từ danh sách trong ngoặc đơn, dùng ' và ' nếu nhiều lựa chọn. "
                "Không giải thích.")
    return f"{inst}\nCâu hỏi: {question}\nTrả lời:"

BINARY_NORM = {
    "có": "có", "co": "có", "yes": "có", "true": "có",
    "không": "không", "khong": "không", "no": "không", "false": "không",
}

def postprocess(raw: str, qtype: str, question: str) -> str:
    s = raw.strip().lower()
    if qtype in ("yes_no", "spatial"):
        for k, v in BINARY_NORM.items():
            if k in s:
                return v
        return s[:10]
    elif qtype == "how_many":
        nums = re.findall(r'\d+', s)
        return nums[0] if nums else "0"
    else:
        m = re.search(r'\((.+?)\)$', question)
        if m:
            opts = [o.strip() for o in m.group(1).split(',')]
            found = [o for o in opts if o.lower() in s]
            if found:
                return " và ".join(found)
        return s[:60]

# ── Build messages theo chuẩn chat của Qwen ──────────────────
def make_messages(image, prompt, answer=None):
    user = {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": prompt},
    ]}
    if answer is None:
        return [user]
    return [user, {"role": "assistant", "content": [{"type": "text", "text": answer}]}]

# ── Dataset ──────────────────────────────────────────────────
class VQADataset(Dataset):
    def __init__(self, csv_path, images_dir, processor, max_rows=None):
        df = pd.read_csv(csv_path)
        if max_rows is not None and len(df) > max_rows:
            df = df.sample(n=max_rows, random_state=42).reset_index(drop=True)
        self.df = df
        self.images_dir = images_dir
        self.processor = processor
        self._img_cache = {}
        self.image_pad_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")

    def __len__(self):
        return len(self.df)

    def _load_image(self, fn):
        if fn not in self._img_cache:
            try:
                img = Image.open(os.path.join(self.images_dir, fn)).convert("RGB")
            except Exception:
                img = Image.new("RGB", (336, 336))
            if len(self._img_cache) >= MAX_IMG_CACHE:
                self._img_cache.pop(next(iter(self._img_cache)))
            self._img_cache[fn] = img
        return self._img_cache[fn]

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        image  = self._load_image(row["filename"])
        prompt = build_prompt(row["question"], row["question_type"])
        answer = str(row["answer"]).strip()

        msgs_full   = make_messages(image, prompt, answer)
        msgs_prompt = make_messages(image, prompt, None)

        text_full   = self.processor.apply_chat_template(
            msgs_full, tokenize=False, add_generation_prompt=False)
        text_prompt = self.processor.apply_chat_template(
            msgs_prompt, tokenize=False, add_generation_prompt=True)

        # Xử lý ảnh CHỈ 1 LẦN
        image_inputs, _ = process_vision_info(msgs_full)
        enc = self.processor(text=[text_full], images=image_inputs,
                             padding=False, return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)

        # prompt_len trong chuỗi đã expand image-pad, không cần chạy lại image_processor:
        # text_prompt là tiền tố của text_full -> tokenize bằng tokenizer (rẻ),
        # rồi thay 1 placeholder <|image_pad|> bằng số image-pad thực tế đã expand.
        num_img    = int((input_ids == self.image_pad_id).sum())
        prompt_ids = self.processor.tokenizer(text_prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids) - 1 + num_img

        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      input_ids,
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   enc["pixel_values"],        # (num_patches, dim) — KHÔNG squeeze
            "image_grid_thw": enc["image_grid_thw"],      # (1, 3)
            "labels":         labels,
        }

# ── Collator: pad text động + nối pixel_values/grid theo dim 0 ─
def make_collate_fn(processor):
    pad_id = processor.tokenizer.pad_token_id

    def collate_fn(batch):
        maxlen = max(b["input_ids"].shape[-1] for b in batch)
        input_ids, attn, labels = [], [], []
        for b in batch:
            n = b["input_ids"].shape[-1]; p = maxlen - n
            input_ids.append(torch.cat([b["input_ids"],
                              torch.full((p,), pad_id, dtype=b["input_ids"].dtype)]))
            attn.append(torch.cat([b["attention_mask"],
                          torch.zeros(p, dtype=b["attention_mask"].dtype)]))
            labels.append(torch.cat([b["labels"],
                            torch.full((p,), -100, dtype=b["labels"].dtype)]))
        return {
            "input_ids":      torch.stack(input_ids),
            "attention_mask": torch.stack(attn),
            "labels":         torch.stack(labels),
            # Qwen: pixel_values là danh sách patch -> NỐI theo dim 0, không stack
            "pixel_values":   torch.cat([b["pixel_values"] for b in batch], dim=0),
            "image_grid_thw": torch.cat([b["image_grid_thw"] for b in batch], dim=0),
        }
    return collate_fn

# ── Load model — mỗi rank nạp full model vào GPU của nó ───────
rprint(f"[DDP] WORLD_SIZE={WORLD_SIZE} | loading Qwen2.5-VL-3B on each GPU...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.float16,
    device_map={"": f"cuda:{LOCAL_RANK}"}, low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
    # Chỉ các lớp này có trong LLM của Qwen; vision tower dùng tên khác nên không bị đụng
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
if RANK == 0:
    model.print_trainable_parameters()
rprint("Model loaded.")

# ── Dataset ──────────────────────────────────────────────────
rprint("Building dataset...")
train_ds = VQADataset(TRAIN_CSV, IMAGES_DIR, processor, max_rows=MAX_TRAIN_ROWS)
rprint(f"Train: {len(train_ds)} rows (chia cho {WORLD_SIZE} GPU)")

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
    label_names=["labels"],
    ddp_find_unused_parameters=False,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds,
                  data_collator=make_collate_fn(processor))

rprint("Starting fine-tuning...")
trainer.train()
rprint("Fine-tuning done.")

if RANK == 0:
    model.save_pretrained(OUT_DIR)
    processor.save_pretrained(OUT_DIR)
    print(f"Adapter saved -> {OUT_DIR}")

# ── Đồng bộ rồi tắt DDP; rank != 0 thoát, chỉ rank 0 đi eval ──
if WORLD_SIZE > 1:
    torch.distributed.barrier()
    torch.distributed.destroy_process_group()
if RANK != 0:
    sys.exit(0)

# ============================================================
# EVALUATION — chỉ rank 0, single-GPU
# ============================================================
print("\nEvaluating on test set...")
model = trainer.model
model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()
processor.tokenizer.padding_side = "left"

df_test = pd.read_csv(TEST_CSV)
done = set()
rows_out = []
if os.path.exists(CKPT_CSV):
    df_ckpt = pd.read_csv(CKPT_CSV)
    rows_out = df_ckpt.to_dict("records")
    done = {(r["filename"], r["qid"]) for r in rows_out}
    print(f"Resume: {len(done)} rows done.")

img_cache = {}

def infer_ft(image, prompt: str) -> str:
    msgs = make_messages(image, prompt, None)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msgs)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=48, do_sample=False)
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(gen_ids, skip_special_tokens=True).strip()

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluating"):
    fn  = row["filename"]
    qid = row["qid"]
    if (fn, qid) in done:
        continue

    if fn not in img_cache:
        try:
            img_cache[fn] = Image.open(os.path.join(IMAGES_DIR, fn)).convert("RGB")
        except Exception:
            img_cache[fn] = None
        if len(img_cache) > 50:
            del img_cache[next(iter(img_cache))]

    image = img_cache.get(fn)
    if image is None:
        pred = "error"
    else:
        prompt = build_prompt(row["question"], row["question_type"])
        try:
            raw  = infer_ft(image, prompt)
            pred = postprocess(raw, row["question_type"], row["question"])
        except Exception:
            pred = "error"

    rows_out.append({
        "filename":      fn,
        "qid":           qid,
        "question_type": row["question_type"],
        "ground_truth":  str(row["answer"]).strip().lower(),
        "prediction":    pred,
    })
    if len(rows_out) % 200 == 0:
        pd.DataFrame(rows_out).to_csv(CKPT_CSV, index=False)

# ── Save & report ────────────────────────────────────────────
df_res = pd.DataFrame(rows_out)
df_res.to_csv(OUT_CSV, index=False)

print("\n" + "="*55)
print("MODEL: Qwen2.5-VL-3B  |  QLoRA fine-tuned")
print("="*55)
print(f"{'QID':<5} {'type':<12} {'n':>5}  {'Exact Match':>11}")
print("-"*55)

total_correct = total_n = 0
summary_per_qid = {}
for qid in sorted(df_res["qid"].unique()):
    sub   = df_res[df_res["qid"] == qid]
    gt    = sub["ground_truth"].str.strip().str.lower()
    pr    = sub["prediction"].str.strip().str.lower()
    n     = len(sub); exact = (gt == pr).sum()
    total_correct += exact; total_n += n
    qtype = sub["question_type"].iloc[0]
    print(f"{qid:<5} {qtype:<12} {n:>5}  {exact/n*100:>10.1f}%")
    summary_per_qid[qid] = round(exact / n, 4)

print("-"*55)
print(f"{'OVERALL':<18} {total_n:>5}  {total_correct/total_n*100:>10.1f}%")

summary = {"model": "Qwen2.5-VL-3B", "mode": "QLoRA-finetune",
           "lora_r": LORA_R, "epochs": TRAIN_EPOCHS, "lr": LR,
           "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS,
           "train_rows": len(train_ds),
           "overall_exact": round(total_correct / total_n, 4),
           "per_qid": summary_per_qid}
with open("/kaggle/working/summary_qwen.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Done.")


Writing qwen_train.py


## Cell 3 — Train (DDP 2 GPU) + Eval

In [3]:
!torchrun --nproc_per_node=2 qwen_train.py

W0629 01:12:32.652000 114 torch/distributed/run.py:852] 
W0629 01:12:32.652000 114 torch/distributed/run.py:852] *****************************************
W0629 01:12:32.652000 114 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0629 01:12:32.652000 114 torch/distributed/run.py:852] *****************************************
[DDP] WORLD_SIZE=2 | loading Qwen2.5-VL-3B on each GPU...
preprocessor_config.json: 100%|███████████████| 350/350 [00:00<00:00, 2.13MB/s]
chat_template.json: 1.05kB [00:00, 3.70MB/s]
config.json: 1.37kB [00:00, 5.75MB/s]
tokenizer_config.json: 5.70kB [00:00, 17.3MB/s]
vocab.json: 2.78MB [00:00, 51.6MB/s]
merges.txt: 1.67MB [00:00, 127MB/s]
tokenizer.json: 7.03MB [00:00, 129MB/s]
model.safetensors.index.json: 65.4kB [00:00, 114MB/s]
Fetching 2 files: 100%|██████████████████████

## Cell 4 — Đóng gói model + kết quả để tải về
Tạo `qwen_finetuned.zip` (adapter LoRA + CSV + summary). Tải ở panel **Output** hoặc link bên dưới.

In [4]:
import os, shutil
from IPython.display import FileLink

WORK = "/kaggle/working"
BUNDLE = os.path.join(WORK, "qwen_bundle")
ADAPTER = os.path.join(WORK, "qwen_lora")

if os.path.exists(BUNDLE):
    shutil.rmtree(BUNDLE)
os.makedirs(BUNDLE, exist_ok=True)

assert os.path.exists(ADAPTER), f"Khong thay adapter o {ADAPTER} — da chay Cell 3 xong chua?"
shutil.copytree(ADAPTER, os.path.join(BUNDLE, "qwen_lora"))

for fname in ["results_qwen_ft.csv", "summary_qwen.json"]:
    src = os.path.join(WORK, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(BUNDLE, fname))

zip_path = shutil.make_archive(os.path.join(WORK, "qwen_finetuned"), "zip", BUNDLE)
print(f"Da tao: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
FileLink("qwen_finetuned.zip")

Da tao: /kaggle/working/qwen_finetuned.zip  (550.5 MB)


/kaggle/working/qwen_finetuned.zip

## (tuỳ chọn) Kiểm tra 2 GPU — chạy khi Cell 3 đang train

In [ ]:
!nvidia-smi

Mon Jun 29 04:11:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----